# 🛰️ GalaxEye Satellite Tile Classifier — T4 GPU Training & Hugging Face Safetensors Export

This notebook fine-tunes a **ResNet-18** model on the EuroSAT 7-class satellite tile dataset.
- **Optimized for NVIDIA T4 GPU** (Mixed Precision `fp16` via `torch.cuda.amp`, cuDNN benchmark)
- **Automatic `.zip` Extraction**: Automatically unpacks `Galaxeye-BE_MLSys-TakeHome_Assignment-Tiles.zip`
- **Hugging Face `safetensors` Format**: Saves `model.safetensors` + `config.json` (secure, fast zero-copy loading, ready for Hugging Face Hub upload)

In [ ]:
# 1. Unzip Dataset & Install safetensors if needed
import os
import sys
import zipfile
import glob

try:
    import safetensors
except ImportError:
    print("Installing safetensors...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "safetensors"])
    import safetensors

# Look for zip file in current directory or parent directory
zip_candidates = glob.glob("*.zip") + glob.glob("../*.zip")
print(f"Found zip files: {zip_candidates}")

DATASET_DIR = "be-mlsys-assignment-dataset"

if not os.path.exists(DATASET_DIR) and not os.path.exists("candidate_tiles"):
    zip_path = None
    for z in zip_candidates:
        if "Galaxeye" in z or "Tiles" in z or "assignment" in z.lower():
            zip_path = z
            break
    if not zip_path and zip_candidates:
        zip_path = zip_candidates[0]
        
    if zip_path:
        print(f"📦 Extracting {zip_path}...")
        with zipfile.ZipFile(zip_path, "r") as zip_ref:
            zip_ref.extractall(".")
        print("✅ Extraction complete!")
    else:
        print("⚠️ No zip file found. Please upload Galaxeye-BE_MLSys-TakeHome_Assignment-Tiles.zip")
else:
    print("✅ Dataset folder already extracted.")

In [ ]:
# 2. GPU Detection & NVIDIA T4 Optimizations
import torch
import torch.nn as nn
import torch.optim as optim

if torch.cuda.is_available():
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"⚡ GPU Detected: {gpu_name} ({vram_gb:.2f} GB VRAM)")
    
    # Enable cuDNN autotuner for T4 Tensor Cores
    torch.backends.cudnn.benchmark = True
    USE_AMP = True  # Enable Mixed Precision (FP16) for fast T4 training
    print("🚀 T4 Mixed Precision (AMP) & cuDNN Benchmark Enabled")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = torch.device("mps")
    USE_AMP = False
    print("⚡ Using Apple Silicon MPS")
else:
    device = torch.device("cpu")
    USE_AMP = False
    print("⚠️ Running on CPU")

In [ ]:
# 3. Resolve Dataset Paths
def find_path(rel_name):
    options = [
        os.path.join(DATASET_DIR, rel_name),
        rel_name,
        os.path.join(".", rel_name)
    ]
    for opt in options:
        if os.path.exists(opt):
            return opt
    return os.path.join(DATASET_DIR, rel_name)

CANDIDATE_DIR = find_path("candidate_tiles")
EVAL_DIR = find_path("eval_set")
EVAL_CSV = find_path("eval_labels.csv")

print(f"Candidate tiles directory : {CANDIDATE_DIR} (exists: {os.path.exists(CANDIDATE_DIR)})")
print(f"Eval tiles directory      : {EVAL_DIR} (exists: {os.path.exists(EVAL_DIR)})")
print(f"Eval ground truth CSV     : {EVAL_CSV} (exists: {os.path.exists(EVAL_CSV)})")

In [ ]:
# 4. Transforms, Augmentation & DataLoaders
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split

# Data Augmentation tailored for Satellite Tiles
TRAIN_TRANSFORM = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

VAL_TRANSFORM = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# Load candidate dataset
full_dataset = datasets.ImageFolder(CANDIDATE_DIR, transform=TRAIN_TRANSFORM)
classes = full_dataset.classes
id2label = {i: c for i, c in enumerate(classes)}
label2id = {c: i for i, c in enumerate(classes)}

print(f"Loaded {len(full_dataset)} images across {len(classes)} classes:")
for idx, cls in enumerate(classes):
    print(f"  [{idx}] {cls}")

# 85% Train / 15% Validation split
train_size = int(0.85 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_data, val_data = random_split(full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

BATCH_SIZE = 32
NUM_WORKERS = 2
PIN_MEMORY = (device.type == "cuda")

train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)
val_loader = DataLoader(val_data, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

print(f"\nTrain batch count: {len(train_loader)} ({len(train_data)} samples)")
print(f"Val batch count  : {len(val_loader)} ({len(val_data)} samples)")

In [ ]:
# 5. ResNet-18 Model Setup
print("🧠 Loading Pretrained ResNet-18...")
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Replace final fully connected layer for the 7 EuroSAT classes
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, len(classes))
model = model.to(device)

# Hyperparameters
EPOCHS = 15
LEARNING_RATE = 5e-4
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

# Gradient Scaler for T4 Mixed Precision (FP16)
scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
print(f"Model initialized on {device}. Ready for training.")

In [ ]:
# 6. Training & Validation Loop (T4 Accelerated)
import time
import copy

best_acc = 0.0
best_weights = copy.deepcopy(model.state_dict())

start_time = time.time()
print(f"🚀 Starting {EPOCHS} Epochs on {device} (AMP={USE_AMP})...")
print("=" * 65)

for epoch in range(EPOCHS):
    # --- Training Phase ---
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0
    for inputs, labels in train_loader:
        inputs = inputs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        train_loss += loss.item() * inputs.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += labels.size(0)
        
    scheduler.step()
    ep_train_loss = train_loss / train_total
    ep_train_acc = (train_correct / train_total) * 100.0
    
    # --- Validation Phase ---
    model.eval()
    val_loss, val_correct, val_total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs = inputs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                
            val_loss += loss.item() * inputs.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)
            
    ep_val_loss = val_loss / val_total
    ep_val_acc = (val_correct / val_total) * 100.0
    
    if ep_val_acc > best_acc:
        best_acc = ep_val_acc
        best_weights = copy.deepcopy(model.state_dict())
        marker = "⭐ (Best Val)"
    else:
        marker = ""
        
    print(f"Epoch {epoch+1:02d}/{EPOCHS:02d} | "
          f"Train Loss: {ep_train_loss:.4f} Acc: {ep_train_acc:.1f}% | "
          f"Val Loss: {ep_val_loss:.4f} Acc: {ep_val_acc:.1f}% {marker}")

elapsed = time.time() - start_time
print("=" * 65)
print(f"🎉 Completed in {elapsed:.1f}s | Best Val Accuracy: {best_acc:.2f}%")

# Load best weights
model.load_state_dict(best_weights)

In [ ]:
# 7. Evaluation against Holdout Eval Set (`eval_labels.csv`)
import csv
from PIL import Image

if os.path.exists(EVAL_CSV) and os.path.exists(EVAL_DIR):
    print("📊 Evaluating on holdout eval_set...")
    model.eval()
    
    ground_truth = {}
    with open(EVAL_CSV, mode="r") as f:
        for row in csv.DictReader(f):
            ground_truth[row["filename"]] = row["true_label"]
            
    correct, total = 0, 0
    per_class = {c: {"correct": 0, "total": 0} for c in classes}
    
    with torch.no_grad():
        for filename, true_label in ground_truth.items():
            img_path = os.path.join(EVAL_DIR, filename)
            if not os.path.exists(img_path):
                continue
            img = Image.open(img_path).convert("RGB")
            tensor = VAL_TRANSFORM(img).unsqueeze(0).to(device)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                outputs = model(tensor)
            pred_label = classes[outputs.argmax(1).item()]
            
            is_match = (pred_label == true_label)
            correct += int(is_match)
            total += 1
            
            if true_label in per_class:
                per_class[true_label]["total"] += 1
                if is_match:
                    per_class[true_label]["correct"] += 1
                    
    overall = (correct / total * 100) if total else 0
    print(f"\n🎯 Overall Holdout Accuracy: {correct}/{total} ({overall:.2f}%)\n")
    print("Per-Class Performance on Eval Set:")
    for c, stat in per_class.items():
        c_tot, c_cor = stat["total"], stat["correct"]
        pct = (c_cor / c_tot * 100) if c_tot else 0
        print(f"  • {c:<12}: {c_cor:2d}/{c_tot:2d} ({pct:5.1f}%)")
else:
    print("⚠️ eval_set or eval_labels.csv not found.")

In [ ]:
# 8. Export in Hugging Face Safetensors Format
import json
from safetensors.torch import save_file

# 1. Save weights in safetensors format (CPU mapped for fast offline serving)
cpu_state_dict = {k: v.cpu().contiguous() for k, v in model.state_dict().items()}
save_file(cpu_state_dict, "model.safetensors")

# 2. Save Hugging Face standard config.json metadata
config_dict = {
    "architectures": ["ResNetForImageClassification"],
    "model_type": "resnet",
    "num_labels": len(classes),
    "id2label": {str(i): c for i, c in enumerate(classes)},
    "label2id": {c: i for i, c in enumerate(classes)},
    "image_size": 64,
    "framework": "pytorch",
    "dataset": "EuroSAT-7-subset"
}

with open("config.json", "w") as f:
    json.dump(config_dict, f, indent=2)

# Also save legacy model.pth for backward compatibility
torch.save({"model_state_dict": cpu_state_dict, "classes": classes}, "model.pth")

print("✅ Successfully exported Hugging Face artifacts:")
print("   • model.safetensors")
print("   • config.json")
print("   • model.pth (fallback)")

# Colab download helper
print("\nIf using Google Colab, download files using:")
print("from google.colab import files")
print("files.download('model.safetensors')")
print("files.download('config.json')")

In [ ]:
# 9. (Optional) Direct Upload to Hugging Face Hub
# Uncomment and run if you want to push directly to your HF repo:
#
# !pip install huggingface_hub
# from huggingface_hub import HfApi, login
# login(token="YOUR_HF_WRITE_TOKEN")
# api = HfApi()
# api.upload_file(path_or_fileobj="model.safetensors", path_in_repo="model.safetensors", repo_id="your-username/galaxeye-tile-classifier", repo_type="model")
# api.upload_file(path_or_fileobj="config.json", path_in_repo="config.json", repo_id="your-username/galaxeye-tile-classifier", repo_type="model")
# print("Uploaded to Hugging Face Hub!")